In [1]:
from MatrixMod2 import *
from Simp import *

In [3]:
d = {0: 'a', 1: 'b'}
d.items()

dict_items([(0, 'a'), (1, 'b')])

## Class for Topological space

In [60]:
class Top:
    ''' A class for representing a finite topological space. 
        ------------
        Attributes |
        ----------------------------------------------------------------
        points: a set {x | x in X } of the points of the top. space
                The elements should all be hashable

        opens: a dictionary of the basic open sets. It is of the form
        { x : { y | x <= y} } where x <= y means x in closure({y})

        closures:   dictionary of the closures of each point
                    { x : cl({x}) }
                    Starts as empty because computing all is O(n^2)
                    Run getClosures() method if all are needed. Else,
                    just call X.cl(x) or X[x] for closure of 1 point
        ----------------------------------------------------------------
    
    '''
    def __init__(self, data, discrete=False, boolean=False, retract=False):
        # can pass a networkx graph as the data
        # import networkx as nx
        ''' The following attributes won't be filled in until computed 
        because they can be expensive to compute for large # of points'''
        self.closures = {}      # can specify closures and ordering,
                                # call self.getClosures()
        self.ordering = None    # but left empty until needed
                                # call self.order()
        self.maxs = None    # max/minimal elements w.r.t. partial ordering
                            # call self.getMaxs()
        self.mins = None    # (must be T_0 space to make sense)
                            # call self.getMins()
        self.T0 = None      # has been checked to be T0
                            # call self.isT0()
        self.beats = None   # points with a unique succesor or predecesor
                            # call self.getBeats()
        if type(data) == dict: # dict of open sets {x: {y | y <= x} }
            self.opens = data
            self.points = set(data)
        
        elif type(data) == int:
            assert data >= 0, 'Topology of integer can\'t be negative'
            assert (discrete and boolean)==False, ('boolean lattice is '
                                                    'not discrete')
            self.points = {n for n in range(data)}            
            if not discrete: 
                if not boolean: # order linearly
                    self.opens = {n: {k for  k in range(n+1)} 
                                for n in range(data)}
                else: # make power set on n elements <= under inclusion
                    # use binary expansion to encode subsets
                    pts = range(1 << data)
                    self.points = {pt for pt in pts}
                    self.opens = {pt: {x for x in pts if x & pt == x}
                                  for pt in pts}

            else: # discrete topology
                self.opens = {n: {n} for n in range(data)}

        elif type(data) == Simp:
            label = lambda x: (
                                x if data.labels is None
                                else tuple(data.as_points(x))
                               ) # express as chains if there is a labelling
            pts = set().union(*[subfaces(max_face) 
                                        for max_face in data.maxs]) # all faces
            self.points = {label(pt) for pt in pts}
            self.opens = {label(pt1): 
                          {label(pt2) for pt2 in pts if data.is_face(pt2, pt1)} 
                               for pt1 in pts}
            if retract: # identify each chain with its maximum
                pts = [chain[-1] for chain in self.points]
                opens = {pt: {chain[-1] for chain in self.opens[pt]}
                          for pt in self.opens}
                opens = {pt: set().union(*[opens[pt1] for pt1 in opens if pt1[-1] == pt])
                                           for pt in pts}
                self.opens = opens
                

            # self.ordering = tuple([label(pt) for pt in sorted(self.points)])
            # # normal order respects inclusion of corresponding chains
            # self.maxs = {label(pt) for pt in data.maxs}
            # self.mins = {label(pt) for pt in data.points}
            # self.T0 = True
            
            

        ''' optional functionality with networkx directed graphs, but I
        currently am not using this, so I don't want to bother importing '''

        # elif type(data) == nx.DiGraph:
        #     opens = {}
        #     for node in data.nodes:
        #         opens[node] = nx.descendants(data,node) | {node}
        #     self.opens = opens
        #     self.points = set(data.nodes)


    def __repr__(self):
        return str(self.opens)
    
    def __len__(self):
        return len(self.points)
    
    def __contains__(self, x):
        return x in self.points
    
    def __iter__(self):
        return iter(self.points)

    def __call__(self, x):
        # Top(x) returns the same as self.opens[x] = {y | y <= x}
        return self.opens[x]
        
    def __getitem__(self, x):
        # Top[x] returns the same as Top.cl(x) or Top.closures[x]
        # This convention mirrors the notation (a,b) for open interval 
        # and [a,b] for closed interval
        try:
            return self.closures[x]
        except:
            self.closures[x] = self.cl(x)
            return self.closures[x]
        
    def __mul__(self, other):
        # product topology
        return self.product(other)
    
    def __add__(self, other):
        # disjoint union topology
        if not self.points & other.points: # already disjoint
            return Top(self.opens | other.opens)
        else: # force them to be disjoint
            point0 = Top({0: {0}})
            point1 = Top({1: {1}})
            return Top((point0 * self).opens | (point1 * other).opens)
        
    def __eq__(self, other):
        return self.opens == other.opens
    
    def __le__(self, other):
        # is a subspace
        if self.points & other.points != self.points:
            return False # isn't a subset
        for point in self.points:
            if other(point) & self.points != self(point):
                return False # not the same open sets
        return True
    
    def __lt__(self, other):
        # proper subspace
        return (not self == other) and self <= other
    
    def __ge__(self, other):
        # contains other as a subspace
        return other <= self
    
    def __gt__(self, other):
        # proper super space
        return (not self == other) and self >= other
    
    def __truediv__(self, subspace):
        # quotient by a subspace
        assert self >= subspace, 'Quotient defined for a subspace'
        quotient = set(self.points - subspace.points)

        # the quotient projection
        def pi(x, pt): 
            if x in self.points - subspace.points:
                return x
            elif x in subspace.points:
                return pt
            else:
                raise ValueError('projection map domain error')
        
        # adjoin a new point not in X \ A, make sure key isn't taken
        pt = '*'; idx = 0
        while pt in quotient:
            idx += 1
            pt = '*' + str(idx)
        # now make the open sets for each x in X \ A
        opens = {x: {pi(y, pt) for y in self(x)} for x in quotient}
        # add in the open set around the new extra point 
        opens[pt] = set()
        for a in subspace.points:
            opens[pt] |= {pi(x, pt) for x in self(a)}
        return Top(opens)
    
    def __pow__(self, other):
        return other.hom(self)

    def __invert__(self):
        return self.op()

    def all_opens(self) -> list[set]:
        ''' Generate the whole topology from the basis'''
        anti = self.anti_chains()
        def genOpen(antichain : set) -> set:
            '''{a, b, c} -> U_a \\cup U_b \\cup U_c '''
            return set().union(*[self(x) for x in antichain])
        
        return [genOpen(chain) for chain in anti]

    def anti_chains(self) -> list:
        ''' returns list of all subsets A s.t. any two elements of A are
        incomparable. '''
        order = self.ordering if self.ordering else self.order()
        antichains = [set()]

        def incomp(x, y) -> bool: # test if two points are incomparable
            return not (x in self(y) or y in self(x))
        
        def extendAntiChain(antichain : set) -> list:
            antichains.append(antichain)
            ind = self.index
            for x in self.points - antichain:
                if all([incomp(x, y) and ind(x)>ind(y) for y in antichain]):
                    # don't add {x, y} and {y, x} separately                    
                    extendAntiChain(antichain | {x})
            return
        
        for pt in order:
            extendAntiChain({pt})
        return antichains

    def cl(self, x):
        # returns the closure of the singleton {x}
        assert x in self.points,\
        f'{x} is not a point in the topological space'
        close = {y for y in self if x in self(y)}
        # close = { y | x <= y } = { y | x in self.opens[y] }
        return close

    def cone(self):
        ''' Returns the (discrete) suspension '''
        pt  = '*' # extra point to attach lines to 
        if pt in self:
            num = 1
            while pt + str(num) in self: 
                num += 1
            pt += str(num)
        # unique str for pt found   
         
        opens = (self.opens).copy() # original space embeds into suspension
        opens[pt] = self.points | {pt} # pt is bigger than everything else
        return Top(opens)

    def core(self, relabel=False): 
        '''Returns a Top space in the same homotopy class with the min
        number of points. Removing a beat point is a retraction'''
        order = self.ordering if self.ordering else self.order()
        core = self
        while True:
            order = core.ordering
            # keep removing beat points until none remain
            for pt in reversed(order): # keep smallest to avoid relabeling
                if core.is_beat(pt):
                    core = core.subspace(core.points - {pt})
                    core.ordering = list(order).remove(pt)
                    continue
            break
        return core.relabel() if relabel else core
    
    def get_beats(self) -> set:
        if self.beats: # already computed
            return self.beats
        beats = set()
        for pt in self.points:
            if len(self.succ(pt)) == 1: beats |= {pt} # unique succesor
            elif len(self.pred(pt)) == 1: beats |= {pt} # unique predecesor
        self.beats = beats
        return self.beats
        
    def get_closures(self):
        # populates Top.closures for all elements
        for x in self:
            try: # don't bother if it's already computed
                self.closures[x]
            except:
                self.closures[x] = self.cl(x)

    def get_maxs(self, checkT0=False):
        '''Returns the set of maximal elements wrt the partial ordering.
        Needs to be T_0 space otherwise can have infinite chains. '''
        if checkT0: # O(n^2) to check, so better to avoid if possible            
            assert self.is_t0(), ('maximal elements aren\'t defined for pre-'
            'orders which aren\'t partial orders') 
        if self.maxs == None: # not already computed
            self.maxs = {x for x in self if self[x] == {x}}
        return self.maxs
    
    def get_mins(self, checkT0=False):
        '''Returns the set of maximal elements wrt the partial ordering.
        Needs to be T_0 space otherwise can have infinite chains. '''
        if checkT0: # O(n^2) to check, so better to avoid if possible            
            assert self.is_t0(), ('maximal elements aren\'t defined for pre-'
            'orders which aren\'t partial orders') 
        if self.mins == None: # not already computed
            self.mins = {x for x in self if self(x) == {x}}
        return self.mins
       
    def grading(self, ordered=False):
        ''' Returns a dictionary {n : {points n levels up from bottom}}
        grading[0]={minimals}, grading[1]={x | y < x ==> y minimal}, etc
        
        If (X, <=) comes from a simplicial complex wrt inclusion, then
        X.grading()[n] is the set of n-faces. If not, then the grading 
        isn't necessarily unique, so '''
        grade = {}
        level = 0
        grade[0] = self.get_mins()
        remaining = self.points - grade[0]
        while remaining != set():
            level += 1
            grade[level] = set()
            for pt in grade[level - 1]:
                grade[level] |= self.succ(pt)
            remaining -= grade[level]
        if ordered:
            order = self.order()
            def index(pt): # pass as key to sorted() to sort by top ordering
                for i, x in enumerate(order):
                    if x == pt:
                        return i
            for n in grade: # return ordered tuples instead of sets
                in_order = sorted(list(grade[n]), key=index)
                grade[n] = tuple(in_order)
        return grade
     
    def hom(self, other, domain_order=False):
        # X.hom(Y) returns the topological space Hom(X, Y) with the com-
        # pact-open topology: f <= g iff f(x) <= g(x) for all x
        funcs = self.to(other)
        opens = {}
        order = self.ordering if self.ordering else self.order()
        def funcTuple(func):
            # use top ordering to write func as (f(x_0), ..., f(x_n))
            # so that it is hashable            
            return tuple([func[x] for x in order])

        for f1 in funcs:
            key = funcTuple(f1)
            opens[key] = {funcTuple(f2) for f2 in funcs
                          if all(f2[x] in other.opens[f1[x]] 
                                        for x in self.points)    
                            }
        if domain_order:
            return order, Top(opens)
        return Top(opens)
  
    def is_beat(self, pt) -> bool:
        if len(self.succ(pt)) == 1: # upbeat
            return True
        return  len(self.pred(pt)) == 1 # true if downbeat else false
   
    def isleq(self, x, y) -> bool:
        # partial order defined by x<=y iff x in self.opens[y]
        # or equivalently iff y in self.closures[x]
        return x in self(y)

    def index(self, pt) -> int: 
        ''' pass as key to sorted() to sort by top ordering '''
        order = self.ordering if self.ordering else self.order()
        for i, x in enumerate(order):
            if x == pt:
                return i
    
    def is_t0(self) -> bool:
        # True iff self.opens[x] = self.opens[y] implies x = y
        if self.T0 == None:
            checked = set()
            for x in self:
                for y in self.points - (checked | {x}):
                    if self(x) == self(y):
                        self.T0 = False
                        return False
                checked |= {x} # don't check U_x=U_y and U_y=U_x separately
            self.T0 = True
        return self.T0
    
    def is_t1(self) -> bool:
        # for finite spaces, this is the same as being discrete
        for x in self:
            if self(x) != {x}:
                return False
        return True

    def lattice(self):
        ''' returns the lattice of all open subsets, i.e. returns a Top
        object L where L.points = self.allOpens and <= is inclusion'''
        pts = [self.sort(chain) for chain in self.anti_chains()]
        def isSubset(chain : tuple, other : tuple) -> bool:
            # determine inclusion of open sets corresponding to antichains
            # \forall x in chain, \exists y in other, s.t. x <= y
            return all([any([self.isleq(x, y) for y in other]) for x in chain])
        opens = {pt: {sub for sub in pts if isSubset(sub, pt)} for pt in pts}
        return Top(opens)
            
    def lower(self, pts : set) -> set:
        ''' given some subset pts of points, return the set 
            lower(pts) := {x | \\exists pt in pts s.t. x <= pt}'''
        return set().union(*[self(pt) for pt in pts])

    def max_chains(self, checkT0=False) -> list[tuple]:
        ''' Returns a list of the maximal chains in a T0 top. sp. '''
        if checkT0:
            assert self.is_t0(), ('maximal chains aren\'t defined for pre-'
            'orders which aren\'t partial orders')
        
        order = self.order() # for consistency of ordering
        def index(pt): # pass as key to sorted() to sort by top ordering
            for i, x in enumerate(order):
                if x == pt:
                    return i
        maxs = sorted(self.get_maxs(), key=index)
        mins = sorted(self.get_mins(), key=index) 
        chains = []

        def extendChain(chain : list) -> None:
            ''' given a non-empty chain, extend it further if possible '''
            curr_max = chain[-1]
            bigger = sorted(self.succ(curr_max), key=index)
            if curr_max in maxs: # already maximal, add to list
                chains.append(tuple(chain))
                return
            for pt in bigger:
                new_chain = chain.copy()
                new_chain.append(pt)
                extendChain(new_chain)
        
        # extend all chains starting at minimal elements
        for min in mins:
            extendChain([min])
        return chains # will be ordered lexographically
  
    def op(self):
        # reverse the ordering
        self.get_closures()
        op_opens = self.closures
        X_op = Top(op_opens)
        X_op.closures = self.opens
        X_op.maxs = self.mins
        X_op.mins = self.maxs
        if self.ordering:
            op_order = list(self.ordering).copy()
            op_order.reverse()
            X_op.ordering = tuple(op_order)
        return (X_op)
    
    def order(self, update=True):
        # returns a topological ordering of self.points, meaning that if
        # x <= y, then x will appear first in the list (but converse may 
        # be false if x and y aren't comparable)

        nodes = {x: {'start':0, 'end':0, 'parent':None, 'visited':False}
                 for x in self.points} # nodes for depth first search
        time = 0 
        def dfsVisit(pt): # one iteration of depth first search
            nonlocal time
            time += 1
            node = nodes[pt]
            node['start'] = time
            node['visited'] = True

            for child in sorted(list(self[pt]), key=hash, reverse=True): # pt <= child
                childnode = nodes[child]
                if childnode['visited'] == False:
                    childnode['parent'] = node
                    dfsVisit(child)
            time += 1
            node['end'] = time
            node['visited'] = 'done'

        # now iterate over all points 
        ordered_pts = sorted(list(self.points), key=hash, reverse=True)
        for pt in ordered_pts:
            if nodes[pt]['visited'] == False:
                dfsVisit(pt)

        # after depth first search, order the points by finish time
        top_order = sorted(nodes, reverse=True,
                           key=lambda x: nodes[x]['end'] )
        if update:
            self.ordering = tuple(top_order)
        return tuple(top_order)
    
    def pred(self, x, ordered=False):
        ''' Given point x in X, returns the immediate predecessors 
            {z in X | z < x and (y<z<=x ==> y=z) }  '''
        assert x in self.points, f'{x} is not a point in the topological space'
        pred = self(x) - {x} # remove x from the set of successors
        for y in pred.copy(): # if y > x
            if y not in pred: # y may have been removed proviously
                continue
            for z in (self(y) - {y}) & pred: # and z > y
                pred -= {z} # z can't be a successor of x
        if ordered:
            pred = self.sort(pred)       
        return pred

    def product(self, other):
        # cartesian product
        def prod(set1, set2):
            return [(x, y) for x in set1 for y in set2]
        ord1 = self.order()
        ord2 = other.order()
        points = prod(ord1, ord2)
        opens = {point : set(prod(self(point[0]), other(point[1]))) \
                 for point in points}
        return Top(opens)
       
    def subspace(self, subset):
        # subspace topology
        assert subset & self.points == subset, 'needs to be a subset'
        return Top({x: self(x) & subset for x in subset})

    def t0_closure(self):
        t0 = self.T0 if self.T0 != None else self.is_t0()
        if t0: return self # already T0

        # self / ~    to do!
   
    def to(self, other, inj=False):
        ''' returns a list of continuous fnunctions self -> other. Each
        function is represented as a dictionary {x: f(x)}.
        X.to(Y) returns [{x: f(x)} for f: X --> Y continuous]'''
        P_ordered = self.order()          # topologically sorted points
        results = []

        # helper function to iteratively modify an existing function
        def backtrack(index, current_map):  
            if index == len(P_ordered): # complete function
                results.append(current_map.copy())
                return 
            x = P_ordered[index]
            for q in other.points: # potential values for f(x)
                valid = True # make sure (x <= y) ==> ( f(x) <= f(y) )
                for y in current_map:
                    fy = current_map[y]
                    if y in self.opens[x] and fy not in other.opens[q]:
                        valid = False
                        break
                if valid:
                    current_map[x] = q # assign f(x) = q
                    backtrack(index + 1, current_map) # progress 1 step
                    # reached end of valid mapping
                    del current_map[x] # remove last assignment, iterate

        backtrack(0, {})
        return results
 
    def relabel(self):
        # returns a homeomorphic top space with integer labels
        order = self.ordering if self.ordering else self.order()
        opens = {n:{m for m in range(len(self)) 
                    if order[m] in self(order[n])}
                for n in range(len(self))}
        return Top(opens)

    def simp(self, labels=True) -> Simp:        
        chains = self.max_chains() # will have called self.order()
        labels = self.ordering if labels else None
        ind = self.index
        max_faces = [bitstonum([ind(pt) for pt in chain]) for chain in chains]
        return Simp(max_faces, labels)

    def sort(self, subset, check_subset=True) -> tuple:
        ''' given a subset A <= X, returns A in sorted order accoring to 
        the topological ordering of X. '''
        if check_subset:
            assert subset & self.points == subset, 'needs to be a subset'
        pts = list(subset)
        return tuple(sorted(pts, key=self.index))
    
    def succ(self, x, ordered=False):
        ''' Given point x in X, returns the immediate successors 
            {z in X | x < z and (x<y<=z ==> y=z) }  '''
        assert x in self.points, f'{x} is not a point in the topological space'
        succ = self[x] - {x} # remove x from the set of successors
        for y in succ.copy(): # if y > x
            if y not in succ: # y may have been removed proviously
                continue
            for z in (self[y] - {y}) & succ: # and z > y
                succ -= {z} # z can't be a successor of x
        if ordered:
            succ = self.sort(succ)       
        return succ
    
    def susp(self, N=1):
        ''' Returns the Nth iterated (discrete) suspension '''
        if N == 0:
            return self
        elif N == 1:
            north, south  = 'N', 'S' # poles to attach lines to 
            if north in self:
                N_num = 1
                while north + str(N_num) in self: 
                    N_num += 1
                north += str(N_num)
            if south in self:
                S_num = 1 
                while south + str(S_num) in self:
                    S_num +=1        
                south += str(S_num)
            # unique str for poles found   
            
            opens = (self.opens).copy() # original space embeds into suspension
            opens[north] = self.points | {north}
            opens[south] = self.points | {south} # both poles maximal
            return Top(opens)
        else:
            assert type(N) == int and N > 0, ('number of times to iterate '
            'suspension must be non-negative integer')
            return self.susp().susp(N-1)


In [61]:
N=1
S = Top({}).susp(N+1).relabel() # weak homotopic to N sphere
S, S.order(), S.max_chains(), S.simp().get_faces(), S.simp().as_points(9)

({0: {0}, 1: {1}, 2: {0, 1, 2}, 3: {0, 1, 3}},
 (0, 1, 2, 3),
 [(0, 2), (0, 3), (1, 2), (1, 3)],
 {0: [1, 2, 4, 8], 1: [5, 6, 9, 10]},
 [0, 3])

In [62]:
Q = Top(S.simp(), retract=True); Q, S

({3: {0, 1, 3}, 2: {0, 1, 2}, 1: {1}, 0: {0}},
 {0: {0}, 1: {1}, 2: {0, 1, 2}, 3: {0, 1, 3}})

In [6]:
X = Top({0: {0}, 1:{0,1}, 2:{0,2}, 3:{0,2,3}, 4:{0,1,2,3,4}, 5:{0,1,5}})
X.anti_chains()

[set(), {0}, {1}, {1, 2}, {1, 3}, {2}, {2, 5}, {3}, {3, 5}, {4}, {4, 5}, {5}]

### Here's a separate class for a simplicial complex

In [7]:
# ''' First, define some funtions for integer to bit conversion '''

# def bitstonum(bits : list[int]) -> int:
#     # given array of which bits are 1 in binary, return the integer
#     return sum([1 << bit for bit in bits])

# def numtobits(n : int) -> list[int]:
#     # given n, return a list of which bits are 1 in the binary
#     index = [i for i in range(n.bit_length()) if (n >> i) & 1]
#     return index

# def subfaces(face : int, nonempty : bool = True) -> set[int]:
#         # given face, return all subfaces
#         pts = numtobits(face)
#         start = 1 if nonempty else 0
#         subs = [numtobits(sub) for sub in range(start, 1 << len(pts))]
#         subface = [[pts[i]   for i in sub] for sub in subs]
#         return {bitstonum(bits) for bits in subface}  

# class Simp:
#     ''' Encodes an abstract simplicial complex using bitwise operations 
#     on integers. For example, the line {{x0}, {x1}, {x0, x1}} would be 
#     {1,10,11} in binary. Only need to feed in the maximal simplices, all
#     others can be infered. Right now only works for simplices with less
#     than 65 vertices. '''
    
#     def __init__(self, max_faces : int | list[int]) -> None:
#         # determine which points occur in the complex
#         total = 0
#         for face in max_faces:
#              total |= face
#         self.points = {1 << bit for bit in numtobits(total)}
#         max_faces = [max_faces] if type(max_faces) == int else max_faces


#         self.dim : int = max([face.bit_count() for face in max_faces]) - 1
#         self.maxs = max_faces
#         self.faces : dict[set[int]] | None = None # compute only if necessary
                    
#     def bd(self, simplex : int, orient : bool = False) -> set[int]:
#          ''' Returns the maximal subfaces of the simplex. For now I don't
#          care about doing the orientation'''

#          bits = [i for i in range(simplex.bit_length()) if (simplex >> i & 1)]
#          if not orient:
#             return {simplex - (1 << bit) for bit in bits}
         
#          else: # if oriented, the orientation is encoded by negative signs
#               return {(-1) ** i * (simplex - (1 << bit)) 
#                       for i, bit in enumerate(bits)}

#     def complex(self, reduced : bool = True,
#                 orient : bool = False) -> dict[np.ndarray]:
#          ''' Returns a a dict {n: d_n}, where d_n is the nth boundary map,
#          expressed as a matrix d_n : F_2{n-dim faces} -> F_2{(n-1)-dim faces}

#          If reduced is True, then the empty set is considered a (-1)-dim face,
#          in which case d_0 = [1, 1, ..., 1]. Otherwise, d_0 = [0, 0, ..., 0]
#          '''
#          num_pts = len(self.points)
#          faces = self.getFaces(ordered=True)
#          bdry = {0: np.full((1,num_pts), reduced, dtype=bool)}
#          if not orient: # +1 = -1 %2  , so we can just do True and False
#             for n in range(1, self.dim + 1):
#                  simplex, face = faces[n], faces[n-1]
#                  dom, rang = len(simplex), len(face)
#                  bdry[n] = np.array([[face[j] in self.bd(simplex[i]) 
#                                 for i in range(dom)] 
#                                 for j in range(rang)], dtype=bool)
#          else: # need to keep track of negative signs
#               for n in range(1, self.dim + 1):
#                    pass
#          return bdry
                              
#     def getFaces(self, ordered : bool = False) -> dict[set | list]: 
#         # compute all faces from maximal ones
#         if self.faces: 
#             if not ordered:
#                 return self.faces
#             else: 
#                 return {n: sorted(list(self.faces[n])) for n in self.faces}
#         ''' a bit faster if these funcs are in local scope'''
#         def numtobits(n : int) -> list[int]:
#             # given n, return a list of which bits are 1 in the binary
#             index = [i for i in range(n.bit_length()) if (n >> i) & 1]
#             return index
#         def bitstonum(bits : list[int]) -> int:
#             # given array of which bits are 1 in binary, return the integer
#             return sum([1 << bit for bit in bits])
#         def subfaces(face : int, nonempty : bool = True) -> set[int]:
#                 # given face, return all subfaces
#                 pts = numtobits(face)
#                 start = 1 if nonempty else 0
#                 subs = [numtobits(sub) for sub in range(start, 1 << len(pts))]
#                 subface = [[pts[i]   for i in sub] for sub in subs]
#                 return {bitstonum(bits) for bits in subface}  
#         faces : list[set] = [subfaces(max_face) for max_face in self.maxs]
#         self.faces : dict = {-1: set(), 0: self.points} # {} is -1 simplex :)
#         for n in range(1, self.dim + 1):
#              self.faces[n] = {k for k in set.union(*faces) 
#                               if k.bit_count()==n+1}
#         if not ordered:
#             return self.faces # {n: {simplices of dimension n}}
#         # else, {n: [simplices of dimension n in increasing order]}
#         return {n: sorted(list(self.faces[n])) for n in self.faces}
            

In [8]:
N = 8
SN = Simp([((1 << N) - 1) - (1<< i) for i in range(N)]) # ~ (n-2)-sphere
homology2(SN.complex(reduced=False),basis=SN.get_faces(ordered=True))

{6: {'generators': [{127, 191, 223, 239, 247, 251, 253, 254}],
  'boundaries': [set()]},
 5: {'generators': [],
  'boundaries': [{63, 95, 111, 119, 123, 125, 126},
   {63, 159, 175, 183, 187, 189, 190},
   {95, 159, 207, 215, 219, 221, 222},
   {111, 175, 207, 231, 235, 237, 238},
   {119, 183, 215, 231, 243, 245, 246},
   {123, 187, 219, 235, 243, 249, 250},
   {125, 189, 221, 237, 245, 249, 252}]},
 4: {'generators': [],
  'boundaries': [{31, 47, 55, 59, 61, 62},
   {31, 79, 87, 91, 93, 94},
   {47, 79, 103, 107, 109, 110},
   {55, 87, 103, 115, 117, 118},
   {59, 91, 107, 115, 121, 122},
   {61, 93, 109, 117, 121, 124},
   {31, 143, 151, 155, 157, 158},
   {47, 143, 167, 171, 173, 174},
   {55, 151, 167, 179, 181, 182},
   {59, 155, 171, 179, 185, 186},
   {61, 157, 173, 181, 185, 188},
   {79, 143, 199, 203, 205, 206},
   {87, 151, 199, 211, 213, 214},
   {91, 155, 203, 211, 217, 218},
   {93, 157, 205, 213, 217, 220},
   {103, 167, 199, 227, 229, 230},
   {107, 171, 203, 227, 233,

In [9]:
def homology(X : Top):
    ''' Assume X is already simplicial complex with <= being inclusion.
    For now i am just doing Z_2 coefficients for simplicity'''
    import numpy as np
    order = X.order()
    faces = X.grading(ordered=True)
    faces = {n: np.array(faces[n]) for n in faces}
    n = len(faces)
    numfaces = [len(faces[j]) for j in range(n)]
    # dictionary delta = boundary maps between faces {n : delta_n}
    delta = {0: np.zeros((1,numfaces[0]), dtype=bool),
             n: np.zeros((numfaces[n-1], 0))} 
    cycle_sets = {0: [{v} for v in faces[0]]} # every vertex maps to 0
    bdry_sets = {0: [set()]} # which means 0 image
    for dim in range(1, n):
        dom, rang = numfaces[dim], numfaces[dim-1]
        simplex, face = faces[dim], faces[dim-1]

        delta[dim] = np.array([[face[j] in X.pred(simplex[i]) 
                                for i in range(dom)] 
                                for j in range(rang)], dtype=bool)
        cycles = ker2(delta[dim])
        bdrys =  im2(delta[dim+1])
        cycle_sets[dim] = [set(faces[dim][cycle]) for cycle in cycles]
        bdry_sets[dim] = [set(faces[dim][bdry]) for bdry in bdrys]
        
    
    return cycle_sets, bdry_sets
        # try scipy.linalg.(orth, null_space and/or qr)
        
    

In [10]:
Z =Top({0:{0},1:{0,1},2:{0,1,2,9},3:{3,7,0},4:{4,3,7,0},5:{5,3,7,0},
        6:{6,4,5,3,7,0},7:{7},8:{8}, 9:{9}}) 
#custom example to test edge cases
Z.core(True)

{0: {0}, 1: {0, 1}, 2: {0, 2}, 3: {3}}

In [11]:
W = Top({0:{0},1:{1},2:{2},3:{3, 0, 1},4:{4, 0, 1},5:{5, 1, 2},6:{6, 1, 2}})
# join of 2 circles